<a href="https://colab.research.google.com/github/ai-tanzil811/Assignment/blob/main/Hybrid_DLP_AI_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Abstract
The increasing sophistication of cyber-attacks, specifically Phishing and Network Intrusions, demands robust Data Leakage Prevention (DLP) systems. A review of 20 high-impact academic papers reveals that traditional standalone machine learning algorithms struggle with high-dimensional, imbalanced data. This report proposes a hybrid, multi-layered AI architecture leveraging feature selection, SMOTE (Synthetic Minority Over-sampling Technique), and ensemble deep learning to mitigate unauthorized data exposure proactively.

## 2. Literature Synthesis & The "Why"
Data leakage typically follows a kill-chain: it begins with an initial compromise (often Phishing) and ends with data exfiltration (Network Intrusion/Malware).

The **Phishing Vector**: Studies by Alhuzali et al. (2025) and Nayak et al. (2025) highlight the effectiveness of Deep Learning and Feature Selection in detecting malicious URLs and emails. Feature selection is critical for reducing computational overhead while maintaining high accuracy.

The **Exfiltration Vector**: Network traffic logs are highly imbalanced (millions of normal packets vs. a few malicious ones). Ahmed et al. (2022) successfully demonstrated that oversampling techniques (SMOTE) drastically improve detection rates. Furthermore, Talukder et al. (2022) and Su et al. (2020) validate that hybrid and deep learning (DL) models outperform single algorithms on datasets like NSL-KDD.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # Import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

def create_mock_data():
    """
    Generates a synthetic dataset simulating Network Flows and URL Text.
    In a real scenario, replace this with CIC-IDS-2017 or Phishing URL datasets.
    """
    np.random.seed(42)
    num_samples = 1000

    # Synthetic Numeric Features (e.g., packet size, inter-arrival time, login frequency)
    numeric_features = np.random.rand(num_samples, 5) * 100

    # Synthetic Text Features (e.g., URL text for phishing detection)
    normal_urls = [
        "http://google.com/search?q=safe_query",
        "https://www.wikipedia.org/wiki/Main_Page",
        "https://example.com/legit_page",
        "http://mybank.com/secure_login",
        "https://docs.python.org/3/library/random.html"
    ]
    phishing_urls = [
        "http://phishing.site/login.php?user=admin",
        "https://bad-site.ru/update-password",
        "http://free-gift.pw/claim-now",
        "https://secure-login.co/verify_account",
        "http://evil.biz/download_malware.exe"
    ]

    url_texts = np.array([np.random.choice(normal_urls) for _ in range(num_samples)])
    # Make first 100 samples attacks (phishing)
    attack_indices = np.random.choice(num_samples, 100, replace=False)
    url_texts[attack_indices] = np.random.choice(phishing_urls, 100)

    # Target (0 = Normal, 1 = Attack)
    y = np.zeros(num_samples)
    y[attack_indices] = 1 # First 100 are attacks

    # Make the attack numeric features slightly different to simulate anomalies
    numeric_features[attack_indices] += 50

    df = pd.DataFrame(numeric_features, columns=['packet_size', 'flow_duration', 'login_freq', 'bytes_out', 'error_rates'])
    df['url_text'] = url_texts
    df['label'] = y
    return df

def build_hybrid_model():
    """
    Builds a Stacking Classifier with SMOTE oversampling.
    Base Models: Random Forest & XGBoost.
    Meta Model: Logistic Regression.
    """
    # Define Base Learners
    base_learners = [
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
        ('xgb', xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
    ]

    # Define Stacking Ensemble (Hybrid)
    hybrid_classifier = StackingClassifier(
        estimators=base_learners,
        final_estimator=LogisticRegression(),
        cv=5
    )

    # Preprocessing for numeric and text features
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), ['packet_size', 'flow_duration', 'login_freq', 'bytes_out', 'error_rates']),
            ('text', TfidfVectorizer(max_features=1000), 'url_text')
        ],
        remainder='passthrough' # Keep any other columns not specified
    )

    # Create an Imbalanced-Learn Pipeline
    # 1. Preprocess features (Scale numeric, TF-IDF text)
    # 2. Select Top Features (Dimensionality Reduction)
    # 3. Oversample minority class using SMOTE
    # 4. Train Hybrid Model
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k='all')), # k='all' means use all features after preprocessing
        ('smote', SMOTE(random_state=42)),
        ('classifier', hybrid_classifier)
    ])

    return pipeline

# Removed the main() function to break it into separate, documented cells.

## 3. Proposed Methodology & Model Architecture
To build a state-of-the-art DLP system, we propose a Hybrid Stacking Ensemble Model.

### 3.1. Workflow Diagram
```mermaid
graph TD
    A[Raw Network Traffic & Email Logs] --> B[Data Preprocessing]
    B --> C{Feature Extraction}
    C -->|Text| D[TF-IDF / NLP]
    C -->|Numeric| E[Standard Scaling]
    D --> F[Feature Selection]
    E --> F
    F --> G[SMOTE: Data Balancing]
    G --> H[Hybrid Stacking Model]
    H --> I(Base: Random Forest)
    H --> J(Base: XGBoost)
    I --> K{Meta Learner: Deep Neural Network / LR}
    J --> K
    K --> L[Alert: Phishing / Exfiltration Detected]
    L --> M[Automated Mitigation via API]
```

### 3.2. Step-by-Step Execution
**Data Ingestion**: Aggregate logs from endpoints, proxies, and email gateways.
**Feature Engineering & Selection**: Apply SelectKBest or Information Gain to retain only the most critical features, reducing noise and preventing overfitting.
**Oversampling (SMOTE)**: Generate synthetic data for minority attack classes to prevent the model from defaulting to predicting "normal" traffic.
**Hybrid Detection Engine**: Use a stacking classifier where XGBoost and Random Forest parse the selected features, and a Meta-learner makes the final, highly accurate prediction.

## 4. Good Practices for Mitigation
Deploying the AI model is only step one. A holistic DLP strategy requires the following best practices:

**Defense-in-Depth**: AI should complement, not replace, rule-based systems (like firewalls and Zero Trust access controls). If AI confidence is medium, fall back to strict rules.
**Proactive vs. Reactive**: As suggested by Zhang et al. (2025), models should proactively hunt for phishing campaigns based on newly registered domains before the payload is delivered to the user.
**Explainable AI (XAI)**: SOC analysts experience "alert fatigue." Integrating SHAP (SHapley Additive exPlanations) allows the AI to highlight exactly which feature (e.g., unusual data upload size at 3 AM) triggered the alarm.
**Continuous Feedback Loop**: Network environments exhibit "concept drift." Ensure the model is retrained quarterly with updated datasets (e.g., transitioning from NSL-KDD to modern datasets like CIC-IDS2017/2023) to recognize zero-day threats.

## 5. Conclusion
A standalone algorithm is no longer sufficient. By bridging Natural Language Processing for phishing detection and Hybrid Ensemble models for network anomaly detection—fortified by SMOTE and strict feature selection—organizations can dramatically reduce their surface area for data leakage.

## 6. Implementation: Data Loading & Preprocessing

We start by generating a synthetic dataset that mimics real-world network flow and URL text data, including both normal and attack instances. This dataset will be used to train and evaluate our hybrid DLP model. We then split the data into training and testing sets, ensuring a stratified split to maintain the original class distribution.

In [ ]:
# Load Data
print("Loading synthetic network flow and URL text data...")
df = create_mock_data()
X = df.drop('label', axis=1)
y = df['label']

print("Dataset head:")
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts()}")

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Loading synthetic network flow and URL text data...
Dataset head:
   packet_size  flow_duration  login_freq   bytes_out  error_rates  \
0    87.454012     145.071431  123.199394  109.865848    65.601864   
1    15.599452       5.808361   86.617615   60.111501    70.807258   
2    52.058449     146.990985  133.244264   71.233911    68.182497   
3    68.340451      80.424224  102.475643   93.194502    79.122914   
4   111.185289      63.949386   79.214465   86.636184    95.606998   

                                    url_text  label  
0        https://bad-site.ru/update-password    1.0  
1             https://example.com/legit_page    0.0  
2       http://evil.biz/download_malware.exe    1.0  
3  http://phishing.site/login.php?user=admin    1.0  
4        https://bad-site.ru/update-password    1.0  

Dataset shape: (1000, 7)
Class distribution:
label
0.0    900
1.0    100
Name: count, dtype: int64

Training set shape: (700, 6)
Testing set shape: (300, 6)


## 7. Model Training

This section involves building and training our hybrid stacking ensemble model. The `build_hybrid_model()` function constructs a pipeline that includes:

1.  **Preprocessing**: Standard Scaling for numerical features and TF-IDF for text features using `ColumnTransformer`.
2.  **Feature Selection**: `SelectKBest` to choose the most relevant features.
3.  **Oversampling**: `SMOTE` to address class imbalance by generating synthetic samples for the minority class (attacks).
4.  **Hybrid Classifier**: A `StackingClassifier` with Random Forest and XGBoost as base learners, and Logistic Regression as the meta-learner.

The model is then trained on the preprocessed and balanced training data.

In [ ]:
# Build Model Pipeline
print("Building Hybrid Model with ColumnTransformer, Feature Selection, and SMOTE...")
model_pipeline = build_hybrid_model()

# Train Model
print("Training Model (this may take a moment)...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")

Building Hybrid Model with ColumnTransformer, Feature Selection, and SMOTE...
Training Model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model training complete.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 8. Model Evaluation

After training, the model's performance is evaluated on the unseen test dataset. We use standard classification metrics such as the confusion matrix and classification report to assess its ability to correctly identify normal and attack instances. The `classification_report` provides precision, recall, f1-score, and support for each class, which are crucial for evaluating performance on imbalanced datasets.

In [ ]:
# Evaluate Model
print("\n--- Model Evaluation ---")
predictions = model_pipeline.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Normal (0)', 'Data Leak/Attack (1)']))

print("\nPipeline successfully executed. Model ready for potential deployment or further refinement.")


--- Model Evaluation ---

Confusion Matrix:
[[270   0]
 [  0  30]]

Classification Report:
                      precision    recall  f1-score   support

          Normal (0)       1.00      1.00      1.00       270
Data Leak/Attack (1)       1.00      1.00      1.00        30

            accuracy                           1.00       300
           macro avg       1.00      1.00      1.00       300
        weighted avg       1.00      1.00      1.00       300


Pipeline successfully executed. Model ready for potential deployment or further refinement.
